In [ ]:
# Install necessary libraries
!pip install torch torchvision torchaudio transformers scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_scheduler
from tqdm import tqdm
from torch.optim import AdamW
import os
import json
from datetime import datetime


In [ ]:
# ============================================================================
# STEP 1: CONFIGURATION
# ============================================================================
class Config:
    # Paths
    DATA_PATH = "/content/drive/MyDrive/fyp_data/fr_nfr.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/FYP_model"
    CHECKPOINT_DIR = "/content/drive/MyDrive/FYP_checkpoints"

    # Model
    MODEL_NAME = "roberta-base"
    MAX_LENGTH = 128

    # Training
    BATCH_SIZE = 16
    NUM_EPOCHS = 4
    LEARNING_RATE = 2e-5
    WARMUP_STEPS = 0

    # Validation
    VAL_SIZE = 0.1  # 10% of training data for validation
    TEST_SIZE = 0.2

    # Checkpoint
    SAVE_EVERY_N_STEPS = 100  # Save checkpoint every N steps
    SAVE_BEST_ONLY = True

    RANDOM_STATE = 42

config = Config()

# Create directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

# ============================================================================
# STEP 2: DATA LOADING AND PREPROCESSING
# ============================================================================
print("=" * 80)
print("STEP 2: LOADING AND PREPROCESSING DATA")
print("=" * 80)

# Load dataset
df = pd.read_csv(config.DATA_PATH)

# ============================================================================
# STEP 3: VIEW FIRST FEW ROWS
# ============================================================================
print("\n" + "=" * 80)
print("FIRST 10 ROWS OF RAW DATA")
print("=" * 80)
print(df.head(10))

STEP 2: LOADING AND PREPROCESSING DATA

FIRST 10 ROWS OF RAW DATA
                                    Requirement Text Type
0       Third field will be time of task completion.   FR
1  The system shall support file systems with cas...  NFR
2  The software shall recognize gestures like pin...   FR
3  User shall be able to register on STM by provi...   FR
4  The System must allow a user to stipulate whic...  NFR
5  The WCS system shall not allow automatic login...  NFR
6  The product shall be able to display calendar ...  NFR
7  The user shall have the option to sign up usin...   FR
8  For data with enough structuring, web scraping...   FR
9  The system shall allow users to create new sco...   FR


In [ ]:

# Clean column names
df.columns = df.columns.str.strip()

# Rename for consistency
df = df.rename(columns={'Requirement Text': 'requirement_text', 'Type': 'label'})

# Normalize label values
df['label'] = df['label'].str.strip().str.upper()

# Map FR = 0, NFR = 1
df['label'] = df['label'].map({'FR': 0, 'NFR': 1})

# Drop rows with NaN labels
df = df.dropna(subset=['label'])

# Convert labels to integers
df['label'] = df['label'].astype(int)

# Check class distribution
print("\n📊 Class Distribution:")
print(df['label'].value_counts())
print(f"\nTotal samples: {len(df)}")

print("\n" + "=" * 80)
print("FIRST 10 ROWS OF RAW DATA")
print("=" * 80)
print(df.head(10))


📊 Class Distribution:
label
0    3647
1    2992
Name: count, dtype: int64

Total samples: 6639

FIRST 10 ROWS OF RAW DATA
                                    requirement_text  label
0       Third field will be time of task completion.      0
1  The system shall support file systems with cas...      1
2  The software shall recognize gestures like pin...      0
3  User shall be able to register on STM by provi...      0
4  The System must allow a user to stipulate whic...      1
5  The WCS system shall not allow automatic login...      1
6  The product shall be able to display calendar ...      1
7  The user shall have the option to sign up usin...      0
8  For data with enough structuring, web scraping...      0
9  The system shall allow users to create new sco...      0


In [ ]:

# ============================================================================
# STEP 3: TRAIN-VAL-TEST SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("STEP 3: SPLITTING DATA INTO TRAIN, VALIDATION, AND TEST SETS")
print("=" * 80)

# First split: separate test set
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    df['requirement_text'],
    df['label'],
    test_size=config.TEST_SIZE,
    stratify=df['label'],
    random_state=config.RANDOM_STATE
)

# Second split: separate validation from training
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=config.VAL_SIZE,
    stratify=train_val_labels,
    random_state=config.RANDOM_STATE
)

print(f"\n✅ Train size: {len(train_texts)}")
print(f"✅ Validation size: {len(val_texts)}")
print(f"✅ Test size: {len(test_texts)}")

# Save split datasets
train_df = pd.DataFrame({'requirement_text': train_texts, 'label': train_labels})
val_df = pd.DataFrame({'requirement_text': val_texts, 'label': val_labels})
test_df = pd.DataFrame({'requirement_text': test_texts, 'label': test_labels})

train_df.to_csv(os.path.join(config.OUTPUT_DIR, "requirements_train.csv"), index=False)
val_df.to_csv(os.path.join(config.OUTPUT_DIR, "requirements_val.csv"), index=False)
test_df.to_csv(os.path.join(config.OUTPUT_DIR, "requirements_test.csv"), index=False)


STEP 3: SPLITTING DATA INTO TRAIN, VALIDATION, AND TEST SETS

✅ Train size: 4779
✅ Validation size: 532
✅ Test size: 1328


In [ ]:

# ============================================================================
# STEP 4: TOKENIZATION
# ============================================================================
print("\n" + "=" * 80)
print("STEP 4: TOKENIZING DATA")
print("=" * 80)

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained(config.MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=config.MAX_LENGTH,
        return_tensors="pt"
    )

print("\n🔄 Tokenizing train set...")
train_encodings = tokenize(train_texts)

print("🔄 Tokenizing validation set...")
val_encodings = tokenize(val_texts)

print("🔄 Tokenizing test set...")
test_encodings = tokenize(test_texts)

print("✅ Tokenization complete!")



STEP 4: TOKENIZING DATA


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


🔄 Tokenizing train set...
🔄 Tokenizing validation set...
🔄 Tokenizing test set...
✅ Tokenization complete!


In [ ]:

# ============================================================================
# STEP 5: DATASET CLASS
# ============================================================================
class ReqDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ReqDataset(train_encodings, train_labels)
val_dataset = ReqDataset(val_encodings, val_labels)
test_dataset = ReqDataset(test_encodings, test_labels)


In [ ]:

# ============================================================================
# STEP 6: MODEL INITIALIZATION
# ============================================================================
print("\n" + "=" * 80)
print("STEP 6: INITIALIZING MODEL")
print("=" * 80)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Using device: {device}")

# Load model
model = RobertaForSequenceClassification.from_pretrained(config.MODEL_NAME, num_labels=2)
model.to(device)

print("✅ Model loaded successfully!")



STEP 6: INITIALIZING MODEL

🖥️  Using device: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!


In [ ]:

# ============================================================================
# STEP 7: DATA LOADERS
# ============================================================================
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE)

# ============================================================================
# STEP 8: OPTIMIZER AND SCHEDULER
# ============================================================================
optimizer = AdamW(model.parameters(), lr=config.LEARNING_RATE)

num_training_steps = config.NUM_EPOCHS * len(train_loader)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=config.WARMUP_STEPS,
    num_training_steps=num_training_steps
)


In [ ]:

# ============================================================================
# STEP 9: VALIDATION FUNCTION
# ============================================================================
def validate(model, val_loader, device):
    """Validate the model and return metrics"""
    model.eval()
    preds, labels, total_loss = [], [], 0

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            total_loss += outputs.loss.item()
            predictions = torch.argmax(outputs.logits, dim=-1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    # Calculate metrics
    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:

# ============================================================================
# STEP 10: CHECKPOINT SAVING FUNCTION
# ============================================================================
def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, metrics, is_best=False):
    """Save model checkpoint with all necessary information"""

    if is_best:
        checkpoint_path = os.path.join(config.CHECKPOINT_DIR, "best_model")
    else:
        checkpoint_path = os.path.join(config.CHECKPOINT_DIR, f"checkpoint_epoch{epoch}_step{step}")

    os.makedirs(checkpoint_path, exist_ok=True)

    # Save model and tokenizer
    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)

    # Save optimizer and scheduler states
    torch.save({
        'epoch': epoch,
        'step': step,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics
    }, os.path.join(checkpoint_path, "training_state.pt"))

    # Save metrics to JSON
    with open(os.path.join(checkpoint_path, "metrics.json"), 'w') as f:
        json.dump(metrics, f, indent=4)

    print(f"💾 Checkpoint saved: {checkpoint_path}")


In [ ]:

# ============================================================================
# STEP 11: TRAINING LOOP WITH VALIDATION AND CHECKPOINTS
# ============================================================================
print("\n" + "=" * 80)
print("STEP 11: TRAINING WITH VALIDATION AND CHECKPOINTS")
print("=" * 80)

# Training history
training_history = {
    'train_loss': [],
    'train_accuracy': [],
    'train_precision': [],
    'train_recall': [],
    'train_f1': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': []
}

best_val_f1 = 0.0
global_step = 0

progress_bar = tqdm(range(num_training_steps), desc="Training")

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")

    # Training phase
    model.train()
    epoch_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_train_loss += loss.item()
        global_step += 1
        progress_bar.update(1)
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Save checkpoint every N steps
        if global_step % config.SAVE_EVERY_N_STEPS == 0:
            print(f"\n📊 Step {global_step}: Running validation...")
            val_metrics = validate(model, val_loader, device)

            print(f"Val Loss: {val_metrics['loss']:.4f} | "
                  f"Val Acc: {val_metrics['accuracy']:.4f} | "
                  f"Val F1: {val_metrics['f1']:.4f}")

            # Save checkpoint
            save_checkpoint(
                model, tokenizer, optimizer, scheduler,
                epoch, global_step, val_metrics, is_best=False
            )

            model.train()  # Back to training mode

    # End of epoch validation
    avg_train_loss = epoch_train_loss / len(train_loader)
    print(f"\n📈 Epoch {epoch + 1} Training Loss: {avg_train_loss:.4f}")

    print(f"\n🔍 Running end-of-epoch validation...")
    val_metrics = validate(model, val_loader, device)

    # Store history
    training_history['train_loss'].append(avg_train_loss)
    training_history['val_loss'].append(val_metrics['loss'])
    training_history['val_accuracy'].append(val_metrics['accuracy'])
    training_history['val_precision'].append(val_metrics['precision'])
    training_history['val_recall'].append(val_metrics['recall'])
    training_history['val_f1'].append(val_metrics['f1'])

    # Print validation results
    print(f"\n📊 Validation Metrics:")
    print(f"  Loss:      {val_metrics['loss']:.4f}")
    print(f"  Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"  Precision: {val_metrics['precision']:.4f}")
    print(f"  Recall:    {val_metrics['recall']:.4f}")
    print(f"  F1 Score:  {val_metrics['f1']:.4f}")

    # Save best model
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        print(f"\n🌟 New best F1 score: {best_val_f1:.4f}")
        save_checkpoint(
            model, tokenizer, optimizer, scheduler,
            epoch, global_step, val_metrics, is_best=True
        )

print(f"\n{'='*80}")
print("✅ TRAINING COMPLETED!")
print(f"{'='*80}")
# ============================================================================
# STEP 12: SAVE TRAINING HISTORY
# ============================================================================
print("\n" + "=" * 80)
print("STEP 12: SAVING TRAINING HISTORY")
print("=" * 80)

history_path = os.path.join(config.OUTPUT_DIR, "training_history.json")
with open(history_path, 'w') as f:
    json.dump(training_history, f, indent=4)

print(f"✅ Training history saved to: {history_path}")


STEP 11: TRAINING WITH VALIDATION AND CHECKPOINTS


Training:   0%|          | 2/1196 [00:14<2:22:26,  7.16s/it, loss=0.4089]


EPOCH 1/4



Training:   8%|▊         | 100/1196 [00:34<06:18,  2.90it/s, loss=0.4541]


📊 Step 100: Running validation...
Val Loss: 0.3425 | Val Acc: 0.8835 | Val F1: 0.8831
💾 Checkpoint saved: /content/drive/MyDrive/FYP_checkpoints/checkpoint_epoch0_step100


...


📊 Step 1000: Running validation...
Val Loss: 0.3309 | Val Acc: 0.8947 | Val F1: 0.8947
💾 Checkpoint saved: /content/drive/MyDrive/FYP_checkpoints/checkpoint_epoch3_step1000


Training:  92%|█████████▏| 1100/1196 [10:38<00:33,  2.89it/s, loss=0.0710]


📊 Step 1100: Running validation...
Val Loss: 0.3337 | Val Acc: 0.8929 | Val F1: 0.8928
💾 Checkpoint saved: /content/drive/MyDrive/FYP_checkpoints/checkpoint_epoch3_step1100


Training: 100%|██████████| 1196/1196 [11:26<00:00,  3.14it/s, loss=0.0252]


📈 Epoch 4 Training Loss: 0.1012

🔍 Running end-of-epoch validation...

📊 Validation Metrics:
  Loss:      0.3337
  Accuracy:  0.8929
  Precision: 0.8928
  Recall:    0.8929
  F1 Score:  0.8928

🌟 New best F1 score: 0.8928
💾 Checkpoint saved: /content/drive/MyDrive/FYP_checkpoints/best_model

✅ TRAINING COMPLETED!

STEP 12: SAVING TRAINING HISTORY
✅ Training history saved to: /content/drive/MyDrive/FYP_model/training_history.json


In [ ]:

# ============================================================================
# STEP 13: FINAL TEST EVALUATION
# ============================================================================
print("\n" + "=" * 80)
print("STEP 13: FINAL TEST EVALUATION")
print("=" * 80)

# Load best model for final evaluation
best_model_path = os.path.join(config.CHECKPOINT_DIR, "best_model")
model = RobertaForSequenceClassification.from_pretrained(best_model_path)
model.to(device)

print("\n🔍 Evaluating on test set...")
model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels.extend(batch['labels'].cpu().numpy())

# Classification report
print("\n" + "=" * 80)
print("FINAL TEST RESULTS")
print("=" * 80)
print(classification_report(labels, preds, target_names=["FR", "NFR"], digits=4))

# Save test results
test_accuracy = accuracy_score(labels, preds)
test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    labels, preds, average='weighted'
)

test_results = {
    'accuracy': float(test_accuracy),
    'precision': float(test_precision),
    'recall': float(test_recall),
    'f1': float(test_f1),
    'classification_report': classification_report(labels, preds, target_names=["FR", "NFR"], output_dict=True)
}

with open(os.path.join(config.OUTPUT_DIR, "test_results.json"), 'w') as f:
    json.dump(test_results, f, indent=4)



STEP 13: FINAL TEST EVALUATION

🔍 Evaluating on test set...


Testing: 100%|██████████| 83/83 [00:06<00:00, 12.33it/s]


FINAL TEST RESULTS
              precision    recall  f1-score   support

          FR     0.9038    0.9137    0.9087       730
         NFR     0.8932    0.8813    0.8872       598

    accuracy                         0.8991      1328
   macro avg     0.8985    0.8975    0.8980      1328
weighted avg     0.8990    0.8991    0.8990      1328



In [ ]:
# ============================================================================
# STEP 14: SAVE FINAL MODEL (SAFE VERSION)
# ============================================================================
import os
import json
from datetime import datetime

print("\n" + "=" * 80)
print("STEP 14: SAVING FINAL MODEL")
print("=" * 80)

# --- Create directories if needed ---
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# --- Save model and tokenizer ---
final_model_path = os.path.join(config.OUTPUT_DIR, "final_model")
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# --- Safely retrieve metrics ---
best_val_f1 = locals().get('best_val_f1', None)
test_accuracy = locals().get('test_accuracy', None)
test_f1 = locals().get('test_f1', None)
best_model_path = locals().get('best_model_path', "N/A")

# --- Save configuration & metrics ---
config_dict = {
    'model_name': config.MODEL_NAME,
    'max_length': config.MAX_LENGTH,
    'batch_size': config.BATCH_SIZE,
    'num_epochs': config.NUM_EPOCHS,
    'learning_rate': config.LEARNING_RATE,
    'best_val_f1': float(best_val_f1) if best_val_f1 is not None else None,
    'test_accuracy': float(test_accuracy) if test_accuracy is not None else None,
    'test_f1': float(test_f1) if test_f1 is not None else None,
    'training_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open(os.path.join(final_model_path, "config.json"), 'w') as f:
    json.dump(config_dict, f, indent=4)

print(f"\n✅ Final model saved to: {final_model_path}")
print(f"✅ Best model saved to: {best_model_path}")

print("\n" + "=" * 80)
print("🎉 ALL STEPS COMPLETED SUCCESSFULLY!")
print("=" * 80)

if best_val_f1 is not None:
    print(f"\n🏆 Best Validation F1: {best_val_f1:.4f}")
if test_accuracy is not None:
    print(f"🏆 Test Accuracy: {test_accuracy:.4f}")
if test_f1 is not None:
    print(f"🏆 Test F1: {test_f1:.4f}")



STEP 14: SAVING FINAL MODEL

✅ Final model saved to: /content/drive/MyDrive/FYP_model/final_model
✅ Best model saved to: /content/drive/MyDrive/FYP_checkpoints/best_model

🎉 ALL STEPS COMPLETED SUCCESSFULLY!
🏆 Test Accuracy: 0.8991
🏆 Test F1: 0.8990


In [ ]:
# ======================================================================
# STEP 15: USER INPUT TESTING (FIXED VERSION)
# ======================================================================
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# --- Path to your trained model folder ---
MODEL_PATH = "/content/drive/MyDrive/FYP_checkpoints/best_model"  # adjust if needed

# --- Load tokenizer ---
tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)

# --- Load model safely, ignoring embedding size mismatches ---
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_PATH,
    ignore_mismatched_sizes=True  # <--- THIS FIXES the 514 vs 512 error
)

# --- Ensure model embeddings are resized to match tokenizer vocab ---
model.resize_token_embeddings(len(tokenizer))

# --- Device setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("\n" + "=" * 80)
print("🧠 PATTERN-BASED REQUIREMENT EXTRACTOR — USER INPUT TESTING")
print("=" * 80)

# --- Label mapping (adjust if reversed in your dataset) ---
label_map = {0: "Functional Requirement (FR)", 1: "Non-Functional Requirement (NFR)"}

# --- Prediction helper function ---
def predict_requirement_type(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred_label = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_label].item()

    return label_map[pred_label], confidence


# --- Interactive loop for user input ---
while True:
    user_text = input("\nEnter requirement text (or 'exit' to quit): ").strip()
    if user_text.lower() == "exit":
        print("\n👋 Exiting user test mode.")
        break
    if not user_text:
        print("⚠️ Please enter a valid requirement sentence.")
        continue

    prediction, confidence = predict_requirement_type(user_text)
    print(f"🔍 Predicted Type → {prediction}")
    print(f"📊 Confidence → {confidence*100:.2f}%")



🧠 PATTERN-BASED REQUIREMENT EXTRACTOR — USER INPUT TESTING

Enter requirement text (or 'exit' to quit): he system shall allow users to update their email address
🔍 Predicted Type → Functional Requirement (FR)
📊 Confidence → 99.57%

Enter requirement text (or 'exit' to quit): The admin shall be able to add, edit, or delete user accounts.
🔍 Predicted Type → Functional Requirement (FR)
📊 Confidence → 98.93%

Enter requirement text (or 'exit' to quit): The mobile app shall synchronize offline data when an internet connection is available.
🔍 Predicted Type → Functional Requirement (FR)
📊 Confidence → 91.93%

Enter requirement text (or 'exit' to quit): Users shall be able to reset their password via a verification email.
🔍 Predicted Type → Functional Requirement (FR)
📊 Confidence → 98.69%

Enter requirement text (or 'exit' to quit): The application shall be available 99.9% of the time.
🔍 Predicted Type → Non-Functional Requirement (NFR)
📊 Confidence → 99.45%

Enter requirement text (or 'exi